## Dataset preparation

In [1]:
import pandas as pd
from collections import Counter
import numpy as np
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [2]:
df = pd.read_json('data/train.json')
df_test = pd.read_json('data/test.json')
df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [ ]:
# Convert interest levels to numerical values (high=2, medium=1, low=0) for modeling
mapping = {'high': 2, 'medium': 1, 'low': 0}

df['interest_level'] = df['interest_level'].map(mapping)

In [ ]:
# Remove price outliers (bottom 1% and top 1%) to prevent extreme values from distorting the model
lower_bound = df['price'].quantile(0.01) 
upper_bound = df['price'].quantile(0.99)
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]
# Apply same outlier removal to test set for consistency
lower_bound = df_test['price'].quantile(0.01) 
upper_bound = df_test['price'].quantile(0.99)
df_test = df_test[(df_test['price'] >= lower_bound) & (df_test['price'] <= upper_bound)]

In [ ]:
# Extract the 'features' column (list of amenities) for processing
new_features = list(df['features'])
features_set = []

# Special characters that need to be removed from feature names for consistent matching
stop_char = [',', ' ', '"', "'", '*']

# Clean each feature string by removing unwanted characters
for i in range(len(new_features)):
    for j in range(len(new_features[i])):
        for char in stop_char:
            new_features[i][j] = new_features[i][j].replace(char, "")
        features_set.append(new_features[i][j])
df['features'] = new_features

# Repeat the same cleaning process for the test set
new_features_test = list(df_test['features'])
features_set_test = []
for i in range(len(new_features_test)):
    for j in range(len(new_features_test[i])):
        for char in stop_char:
            new_features_test[i][j] = new_features_test[i][j].replace(char, "")
        features_set_test.append(new_features_test[i][j])
df_test['features'] = new_features_test

In [6]:
print("Number of unique features: ", len(set(features_set)))

Number of unique features:  1529


In [7]:
feature_count = Counter(features_set)
top_20 = [i[0] for i in feature_count.most_common(20)]
print(*top_20, sep='\n') 

Elevator
HardwoodFloors
CatsAllowed
DogsAllowed
Doorman
Dishwasher
NoFee
LaundryinBuilding
FitnessCenter
Pre-War
LaundryinUnit
RoofDeck
OutdoorSpace
DiningRoom
HighSpeedInternet
Balcony
SwimmingPool
LaundryInBuilding
NewConstruction
Terrace


In [8]:
feature_list = ['bathrooms', 'bedrooms']
for i in range(20):
    name = top_20[i]
    feature_list.append(name)
    df[name] = 0
    df_test[name] = 0
    df.loc[df['features'].apply(lambda x: top_20[i] in x), name] = 1
    df_test.loc[df_test['features'].apply(lambda x: top_20[i] in x), name] = 1
    
len(feature_list)

22

In [9]:
def set_train_test(df, df_test):
    X_train, X_test = df[feature_list], df_test[feature_list]
    y_train, y_test = df['price'], df_test['price']
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = set_train_test(df, df_test)

## Linear Regression

In [10]:
class DeterministicLinearRegression:
    def __init__(self, method='analytical', regularization=None, 
                 alpha=1.0, l1_ratio=0.5, learning_rate=0.01, 
                 n_iter=1000, random_state=21, batch_size=1):

        self.method = method
        self.regularization = regularization
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state
        self.batch_size = batch_size
        self.weights = None
        self.loss_history = []
        
        self.rng = np.random.RandomState(random_state)
    
    def _add_bias_term(self, X):
        return np.c_[np.ones(X.shape[0]), X]
    
    def _compute_gradient(self, X, y, y_pred):
        n_samples = X.shape[0]
        errors = y_pred - y
        
        grad_weights = (2 / n_samples) * X.T @ errors
        
        if self.regularization == 'l2':  # Ridge
            reg_penalty = 2 * self.alpha * self.weights
            reg_penalty[0] = 0  
            grad_weights += reg_penalty
        elif self.regularization == 'l1':  # Lasso
            reg_penalty = self.alpha * np.sign(self.weights)
            reg_penalty[0] = 0  
            grad_weights += reg_penalty
        elif self.regularization == 'elasticnet':
            l1_penalty = self.alpha * self.l1_ratio * np.sign(self.weights)
            l2_penalty = 2 * self.alpha * (1 - self.l1_ratio) * self.weights
            reg_penalty = l1_penalty + l2_penalty
            reg_penalty[0] = 0 
            grad_weights += reg_penalty
        
        return grad_weights
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).flatten()  
        X = self._add_bias_term(X)
        
        n_samples, n_features = X.shape
        
        self.weights = self.rng.randn(n_features) * 0.01
        
        if self.method == 'analytical':
            self._fit_analytical(X, y)
        elif self.method == 'gradient_descent':
            self._fit_gradient_descent(X, y)
        elif self.method == 'stochastic_gradient_descent':
            self._fit_deterministic_sgd(X, y)
        else:
            raise ValueError("Метод должен быть 'analytical', 'gradient_descent' или 'stochastic_gradient_descent'")
    
    def _fit_analytical(self, X, y):
        if self.regularization == 'l1' or self.regularization == 'elasticnet':
            self._fit_deterministic_sgd(X, y)
            return
        
        try:
            if self.regularization == 'l2':  # Ridge
                identity = np.eye(X.shape[1])
                identity[0, 0] = 0  
                self.weights = np.linalg.solve(X.T @ X + self.alpha * identity, X.T @ y)
            else: 
                self.weights = np.linalg.solve(X.T @ X, X.T @ y)
        except np.linalg.LinAlgError:
            if self.regularization == 'l2':
                identity = np.eye(X.shape[1])
                identity[0, 0] = 0
                self.weights = np.linalg.pinv(X.T @ X + self.alpha * identity) @ X.T @ y
            else:
                self.weights = np.linalg.pinv(X.T @ X) @ X.T @ y
            
    def _fit_gradient_descent(self, X, y):
        for i in range(self.n_iter):
            y_pred = X @ self.weights
            grad_weights = self._compute_gradient(X, y, y_pred)
            self.weights -= self.learning_rate * grad_weights
                
    def _fit_deterministic_sgd(self, X, y):
        n_samples, n_features = X.shape
        
        all_indices = self._generate_deterministic_indices(n_samples, self.n_iter)
        
        for i in range(self.n_iter):
            idx = all_indices[i]
            
            X_i = X[idx]
            y_i = y[idx]
            
            y_pred = np.dot(X_i, self.weights)
            error = y_pred - y_i
            
            grad_weights = 2 * error * X_i
            
            if self.regularization == 'l2':
                reg_penalty = 2 * self.alpha * self.weights
                reg_penalty[0] = 0
                grad_weights += reg_penalty
            elif self.regularization == 'l1':
                reg_penalty = self.alpha * np.sign(self.weights)
                reg_penalty[0] = 0
                grad_weights += reg_penalty
            elif self.regularization == 'elasticnet':
                l1_penalty = self.alpha * self.l1_ratio * np.sign(self.weights)
                l2_penalty = 2 * self.alpha * (1 - self.l1_ratio) * self.weights
                reg_penalty = l1_penalty + l2_penalty
                reg_penalty[0] = 0
                grad_weights += reg_penalty
            
            self.weights -= self.learning_rate * grad_weights
    
    def _generate_deterministic_indices(self, n_samples, n_iter):
        indices = []
        for i in range(n_iter):
            effective_seed = self.random_state + i
            temp_rng = np.random.RandomState(effective_seed)
            idx = temp_rng.randint(0, n_samples)
            indices.append(idx)
        return indices
    
    def predict(self, X):
        if self.weights is None:
            raise ValueError("First, call fit()")
        
        X = np.array(X)
        X_with_bias = self._add_bias_term(X)
        return X_with_bias @ self.weights

In [11]:
def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
    
def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
    
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

In [12]:
def clean_results():
    columns = ['model', 'train', 'test']
    result_MAE = pd.DataFrame(columns=columns)
    result_RMSE = pd.DataFrame(columns=columns)
    result_R2 = pd.DataFrame(columns=columns)
    return result_MAE, result_RMSE, result_R2

result_MAE, result_RMSE, result_R2 = clean_results()

def write_results(model, pred_train, pred_test, y_train, y_test):
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)

    rmse_train = root_mean_squared_error(y_train, pred_train)
    rmse_test = root_mean_squared_error(y_test, pred_test)
    
    r2_train = r2_score(y_train, pred_train)
    r2_test = r2_score(y_test, pred_test)

    result_MAE.loc[len(result_MAE)] = [model, mae_train, mae_test]
    result_RMSE.loc[len(result_RMSE)] = [model, rmse_train, rmse_test]
    result_R2.loc[len(result_R2)] = [model, r2_train, r2_test]

## MinMaxScaler

X_scaled = (X - X_min) / (X_max - X_min)

In [13]:
class MyMinMaxScaler:
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.min_ = None
        self.max_ = None
        self.data_min_ = None
        self.data_max_ = None
        self.scale_ = None
        
    def fit(self, X):
        X = np.array(X)
        
        self.data_min_ = np.min(X, axis=0)
        self.data_max_ = np.max(X, axis=0)
        
        self.scale_ = (self.feature_range[1] - self.feature_range[0]) / (
            self.data_max_ - self.data_min_ + np.finfo(float).eps)
        
        self.min_ = self.feature_range[0] - self.data_min_ * self.scale_
        
        return self
    
    def transform(self, X):
        if self.data_min_ is None or self.data_max_ is None:
            raise ValueError("Сначала вызовите fit()!")
            
        X = np.array(X)
        X_scaled = X * self.scale_ + self.min_
        
        return X_scaled
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

## StandardScaler

X_scaled = (X - μ) / σ
- μ среднее значение
- σ стандартное отклонение

In [14]:
class MyStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None  # стандартное отклонение
        self.var_ = None    # дисперсия
        self.n_samples_seen_ = 0
        
    def fit(self, X):
        X = np.array(X)
        
        self.original_ndim_ = X.ndim
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        self.mean_ = np.mean(X, axis=0)
        self.var_ = np.var(X, axis=0, ddof=0)  
        self.scale_ = np.sqrt(self.var_)        

        self.scale_[self.scale_ == 0] = 1.0
        
        self.n_samples_seen_ = X.shape[0]
        
        return self
    
    def transform(self, X):
        if self.mean_ is None or self.scale_ is None:
            raise ValueError("Сначала вызовите fit()!")
            
        X = np.array(X)
        original_ndim = X.ndim
        
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        X_scaled = (X - self.mean_) / self.scale_
        
        if original_ndim == 1:
            return X_scaled.flatten()
        return X_scaled
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

## Polinomial features

In [15]:
df['bathrooms_x10'] = [i**2 for i in list(df['bathrooms'])]
df['bedrooms_x10'] = [i**2 for i in list(df['bedrooms'])]
df['interest_x10'] = [i**2 for i in list(df['interest_level'])]

X_poly = df[['bathrooms_x10','bedrooms_x10','interest_x10']]
y_poly = df['price']

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly, y_poly, test_size=0.33, random_state=21)

X_train_poly

,bathrooms_x10,bedrooms_x10,interest_x10
94375,1.0,4,0
56848,1.0,0,1
42115,1.0,1,0
91589,1.0,1,0
16432,1.0,16,0
...,...,...,...
42218,4.0,9,0
23111,1.0,4,0
15351,4.0,9,0
13731,1.0,1,1


## Naive models

In [16]:
train_mean = df['price'].mean()
train_median = df['price'].median()

test_mean = df_test['price'].mean()
test_median = df_test['price'].median()

df['mean'] = [train_mean for i in range(len(df))]
df['median'] = [train_median for i in range(len(df))]
df_test['mean'] = [test_mean for i in range(len(df_test))]
df_test['median'] = [test_median for i in range(len(df_test))]

## Results

In [17]:
result_MAE, result_RMSE, result_R2 = clean_results()

regularizations = {None: 'MyLinReg', 'elasticnet': 'elastic', 'l1': 'lasso', 'l2': 'ridge'}
scalers = {'': None, '_minmax': MyMinMaxScaler(), '_standardscaler': MyStandardScaler()}
feature_types = ['', '_poly']

for feature_type in feature_types:
    if feature_type == '_poly': 
        X_test, X_train, y_test, y_train = X_test_poly.copy(), X_train_poly.copy(), y_test_poly.copy(), y_train_poly.copy()

    for scaler_type in scalers:

        if feature_type == '_poly' and scaler_type!= '': 
            break
        
        X_test_scaled, X_train_scaled = X_test.copy(), X_train.copy()
        
        if scaler_type != '':
            scaler_x = scalers[scaler_type]
            
            X_train_scaled = scaler_x.fit_transform(X_train_scaled)
            X_test_scaled = scaler_x.transform(X_test_scaled)
        
        for reg in regularizations:
            model_name = regularizations[reg] + scaler_type + feature_type
            
            linreg = DeterministicLinearRegression(
                method='analytical',  #stochastic_gradient_descent
                random_state=21, 
                regularization=reg,
                learning_rate=0.01,
                n_iter=1000,
                alpha=0.1 if reg else 1.0 
            )
            
            linreg.fit(X_train_scaled, y_train)
            
            pred_train = linreg.predict(X_train_scaled)
            pred_test = linreg.predict(X_test_scaled)
            
            write_results(model_name, pred_train, pred_test, y_train, y_test)

X_train, X_test, y_train, y_test = set_train_test(df, df_test)

In [18]:
sklearn_linreg = LinearRegression()

ridge = Ridge(alpha=1.0).fit(X_train, y_train)  
lasso = Lasso(alpha=0.1).fit(X_train, y_train)
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_train, y_train)
sklearn_linreg.fit(X_train, y_train)

pred_test = sklearn_linreg.predict(X_test)
pred_train = sklearn_linreg.predict(X_train)
write_results('Sklearn', pred_train, pred_test, y_train, y_test)

ridge_test = ridge.predict(X_test)
ridge_train = ridge.predict(X_train)
write_results('Sklearn_rigde', ridge_train, ridge_test, y_train, y_test)

lasso_test = lasso.predict(X_test)
lasso_train = lasso.predict(X_train)
write_results('Sklearn_lasso', lasso_train, lasso_test, y_train, y_test)

elastic_test = elastic.predict(X_test)
elastic_train = elastic.predict(X_train)
write_results('Sklearn_elastic', elastic_train, pred_test, y_train, y_test)

write_results('naive_mean', df['mean'], df_test['mean'], y_train, y_test)
write_results('naive_median', df['median'], df_test['median'], y_train, y_test)

In [19]:
result_MAE

,model,train,test
0,MyLinReg,711.791166,716.845780
1,elastic,718.767516,722.903728
2,lasso,716.062532,721.441361
3,ridge,711.790844,716.845389
4,MyLinReg_minmax,711.791166,716.845780
5,elastic_minmax,929.364232,928.553630
6,lasso_minmax,886.544612,885.818946
7,ridge_minmax,711.797224,716.844048
8,MyLinReg_standardscaler,711.791166,716.845780
9,elastic_standardscaler,794.188933,799.603101


In [20]:
result_RMSE

,model,train,test
0,MyLinReg,1035.351576,1226.865399
1,elastic,1076.899640,1183.331583
2,lasso,1054.831459,1216.033291
3,ridge,1035.351576,1226.861177
4,MyLinReg_minmax,1035.351576,1226.865399
5,elastic_minmax,1375.450851,1367.212931
6,lasso_minmax,1309.678919,1302.254634
7,ridge_minmax,1035.351911,1226.369151
8,MyLinReg_standardscaler,1035.351576,1226.865399
9,elastic_standardscaler,1125.179404,1288.524368


In [21]:
result_R2['diff'] = abs(result_R2['train'] - result_R2['test'])
result_R2

,model,train,test,diff
0,MyLinReg,0.580034,0.404680,0.175354
1,elastic,0.545652,0.446179,0.099473
2,lasso,0.564082,0.415146,0.148936
3,ridge,0.580034,0.404684,0.175350
4,MyLinReg_minmax,0.580034,0.404680,0.175354
5,elastic_minmax,0.258811,0.260686,0.001874
6,lasso_minmax,0.328002,0.329269,0.001267
7,ridge_minmax,0.580034,0.405161,0.174872
8,MyLinReg_standardscaler,0.580034,0.404680,0.175354
9,elastic_standardscaler,0.504000,0.343338,0.160662
